# FD001 Baseline Machine Learning for Remaining Useful Life (RUL) Prediction

This notebook builds on the prepared FD001 dataset from `01_data_preparation_eda_degradation_analysis.ipynb`. It trains and compares baseline machine learning regressors — Linear Regression, Random Forest, and Gradient Boosting — for predicting Remaining Useful Life (RUL) from engine operating settings and sensor measurements.

**Objectives**
- Load the processed FD001 training data (with RUL target) and the official FD001 test set with true RUL labels.
- Apply standard C-MAPSS preprocessing choices: dropping non-informative (constant) sensors and capping RUL at a maximum value to reflect the piecewise-linear degradation assumption.
- Split training data by engine (not by row) to avoid leakage between train and validation sets.
- Train baseline Linear Regression, Random Forest, and Gradient Boosting models.
- Evaluate all models on a held-out validation split and on the official FD001 test set using RMSE and MAE.
- Compare model performance and inspect feature importance for the tree-based models.

Hyperparameter tuning, XGBoost, and a deeper comparison against state-of-the-art approaches are handled in `03_advanced_ml_modeling.ipynb`. Sequential/deep learning models (LSTM/GRU) are covered in `04_deep_learning_rul_prediction.ipynb`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")


## 1. Load Prepared Training Data

Loads the FD001 training data with the RUL target constructed in notebook 01. If the processed file isn't found in this session, the raw CMAPSS archive is re-downloaded and the same preparation steps are repeated so this notebook can run standalone.

In [ ]:
DATA_DIR = "/content/cmapss_data"
PROCESSED_PATH = "/content/fd001_train_with_rul.csv"

COL_NAMES = (
    ["engine_id", "cycle", "setting1", "setting2", "setting3"]
    + [f"sensor{i}" for i in range(1, 22)]
)

if not os.path.exists(PROCESSED_PATH):
    !wget -O CMAPSSData.zip "https://github.com/sudhakaran-srinivasan/intelligent-predictive-maintenance/raw/main/datasets/raw/CMAPSSData.zip"
    !unzip -o CMAPSSData.zip -d {DATA_DIR}

    train_fd001 = pd.read_csv(f"{DATA_DIR}/train_FD001.txt", sep=r"\s+", header=None)
    train_fd001.columns = COL_NAMES

    max_cycles = (
        train_fd001.groupby("engine_id")["cycle"].max()
        .reset_index()
        .rename(columns={"cycle": "max_cycle"})
    )
    train_fd001 = train_fd001.merge(max_cycles, on="engine_id", how="left")
    train_fd001["RUL"] = train_fd001["max_cycle"] - train_fd001["cycle"]
    train_fd001.to_csv(PROCESSED_PATH, index=False)
else:
    train_fd001 = pd.read_csv(PROCESSED_PATH)

print("Train shape:", train_fd001.shape)
train_fd001.head()


## 2. Load the Official Test Set and True RUL Labels

The FD001 test trajectories are truncated at some point before failure. The true RUL for the **last** recorded cycle of each test engine is provided separately in `RUL_FD001.txt`. This mirrors how the benchmark is evaluated: given a partial trajectory, predict the remaining life at its final observed cycle.

In [ ]:
test_fd001 = pd.read_csv(f"{DATA_DIR}/test_FD001.txt", sep=r"\s+", header=None)
test_fd001.columns = COL_NAMES

rul_fd001 = pd.read_csv(f"{DATA_DIR}/RUL_FD001.txt", sep=r"\s+", header=None)
rul_fd001.columns = ["RUL"]
rul_fd001["engine_id"] = rul_fd001.index + 1

print("Test shape:", test_fd001.shape)
print("RUL labels shape:", rul_fd001.shape)
rul_fd001.head()


## 3. Feature Selection

Notebook 01 identified six sensors with no variation across the dataset (`sensor1`, `sensor5`, `sensor10`, `sensor16`, `sensor18`, `sensor19`). These carry no predictive signal and are dropped. The remaining features are the three operating settings and the 15 non-constant sensors.

In [ ]:
CONSTANT_SENSORS = ["sensor1", "sensor5", "sensor10", "sensor16", "sensor18", "sensor19"]

setting_cols = ["setting1", "setting2", "setting3"]
sensor_cols = [c for c in train_fd001.columns if c.startswith("sensor") and c not in CONSTANT_SENSORS]

feature_cols = setting_cols + sensor_cols
print(f"Using {len(feature_cols)} features:")
print(feature_cols)


## 4. RUL Capping (Piecewise-Linear Degradation Assumption)

Raw RUL grows unbounded for early cycles, but engines don't show meaningful degradation-related sensor drift right after start-up — RUL isn't really predictable from sensor state that early. A standard C-MAPSS practice is to cap RUL at a fixed ceiling (commonly 125 cycles) so the target better reflects an assumed healthy-then-degrading trajectory, rather than asking the model to distinguish, say, an RUL of 250 from 300 using sensor values that look identical.

In [ ]:
RUL_CAP = 125

train_fd001["RUL_clipped"] = train_fd001["RUL"].clip(upper=RUL_CAP)

plt.figure(figsize=(6, 4))
plt.hist(train_fd001["RUL"], bins=50, alpha=0.5, label="Raw RUL")
plt.hist(train_fd001["RUL_clipped"], bins=50, alpha=0.5, label=f"Clipped at {RUL_CAP}")
plt.legend()
plt.title("Effect of RUL Capping")
plt.xlabel("RUL")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 5. Train / Validation Split (by Engine)

Rows from the same engine are highly correlated across consecutive cycles. Splitting randomly by row would leak information between train and validation sets, since near-identical cycles from the same engine could land on both sides. Instead, the split is done at the **engine level**: entire engines go to either the training set or the validation set.

In [ ]:
engine_ids = train_fd001["engine_id"].unique()
train_ids, val_ids = train_test_split(engine_ids, test_size=0.2, random_state=42)

train_split = train_fd001[train_fd001["engine_id"].isin(train_ids)]
val_split = train_fd001[train_fd001["engine_id"].isin(val_ids)]

X_train = train_split[feature_cols]
y_train = train_split["RUL_clipped"]

X_val = val_split[feature_cols]
y_val = val_split["RUL_clipped"]

print("Train engines:", len(train_ids), "| Train rows:", X_train.shape[0])
print("Val engines:  ", len(val_ids), "| Val rows:  ", X_val.shape[0])


## 6. Prepare the Official Test Set (Last Cycle per Engine)

For each test engine, only the final observed cycle is scored, paired with the true RUL from `RUL_FD001.txt`.

In [ ]:
test_last_cycle = (
    test_fd001.sort_values(["engine_id", "cycle"])
    .groupby("engine_id")
    .tail(1)
    .reset_index(drop=True)
)

test_last_cycle = test_last_cycle.merge(rul_fd001, on="engine_id", how="left")
test_last_cycle["RUL_clipped"] = test_last_cycle["RUL"].clip(upper=RUL_CAP)

X_test = test_last_cycle[feature_cols]
y_test = test_last_cycle["RUL_clipped"]

print("Test engines:", X_test.shape[0])
test_last_cycle[["engine_id", "cycle", "RUL", "RUL_clipped"]].head()


## 7. Feature Scaling

Linear Regression benefits from standardized inputs; the scaler is fit on the training split only and applied to validation and test to avoid leakage. Tree-based models don't strictly need scaling but are run on the same scaled features for a consistent pipeline.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


## 8. Baseline Models

### 8.1 Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)


### 8.2 Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_scaled, y_train)


### 8.3 Gradient Boosting Regressor

In [ ]:
gb = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=42,
)
gb.fit(X_train_scaled, y_train)


## 9. Model Evaluation and Comparison

Each model is scored on both the validation split (held-out engines from the training set) and the official FD001 test set, using RMSE, MAE, and R².

In [ ]:
def evaluate_model(model, X, y):
    preds = model.predict(X)
    rmse = np.sqrt(mean_squared_error(y, preds))
    mae = mean_absolute_error(y, preds)
    r2 = r2_score(y, preds)
    return {"RMSE": rmse, "MAE": mae, "R2": r2}, preds

models = {
    "Linear Regression": lr,
    "Random Forest": rf,
    "Gradient Boosting": gb,
}

results = []
test_predictions = {}

for name, model in models.items():
    val_metrics, _ = evaluate_model(model, X_val_scaled, y_val)
    test_metrics, test_preds = evaluate_model(model, X_test_scaled, y_test)
    test_predictions[name] = test_preds

    results.append({
        "Model": name,
        "Val_RMSE": val_metrics["RMSE"], "Val_MAE": val_metrics["MAE"], "Val_R2": val_metrics["R2"],
        "Test_RMSE": test_metrics["RMSE"], "Test_MAE": test_metrics["MAE"], "Test_R2": test_metrics["R2"],
    })

results_df = pd.DataFrame(results).sort_values("Test_RMSE").reset_index(drop=True)
results_df


In [ ]:
metrics_to_plot = ["Val_RMSE", "Test_RMSE", "Val_MAE", "Test_MAE"]

fig, ax = plt.subplots(figsize=(9, 5))
results_df.set_index("Model")[metrics_to_plot].plot(kind="bar", ax=ax)
ax.set_title("Baseline Model Comparison: RUL Prediction Error")
ax.set_ylabel("Error (cycles)")
ax.set_xticklabels(results_df["Model"], rotation=0)
plt.tight_layout()
plt.show()


## 10. Feature Importance (Tree-Based Models)

Inspects which settings and sensors the Random Forest and Gradient Boosting models rely on most, and checks whether this lines up with the sensors that showed the strongest correlation with RUL in the notebook 01 EDA (`sensor11`, `sensor4`, `sensor12`, `sensor7`, `sensor15`).

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "rf_importance": rf.feature_importances_,
    "gb_importance": gb.feature_importances_,
}).sort_values("rf_importance", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(importance_df["feature"], importance_df["rf_importance"])
axes[0].invert_yaxis()
axes[0].set_title("Random Forest Feature Importance")

order = importance_df.sort_values("gb_importance", ascending=False)
axes[1].barh(order["feature"], order["gb_importance"])
axes[1].invert_yaxis()
axes[1].set_title("Gradient Boosting Feature Importance")

plt.tight_layout()
plt.show()

importance_df


## 11. Predicted vs. Actual RUL on the Test Set

Visualizes how closely each model's predictions track the true RUL for the official test engines. Points near the diagonal line indicate accurate predictions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)

for ax, (name, preds) in zip(axes, test_predictions.items()):
    ax.scatter(y_test, preds, alpha=0.6, edgecolor="k", linewidth=0.3)
    lims = [0, max(y_test.max(), preds.max()) + 5]
    ax.plot(lims, lims, "r--", linewidth=1)
    ax.set_title(name)
    ax.set_xlabel("Actual RUL")
    ax.set_ylabel("Predicted RUL")

plt.tight_layout()
plt.show()


## 12. Save Baseline Results and Models

Persists the comparison table and the fitted models/scaler so notebook 03 (advanced ML) can load them as a reference baseline rather than retraining from scratch.

In [ ]:
os.makedirs("/content/reports", exist_ok=True)
os.makedirs("/content/models", exist_ok=True)

results_df.to_csv("/content/reports/baseline_model_comparison.csv", index=False)

joblib.dump(scaler, "/content/models/baseline_scaler.pkl")
joblib.dump(lr, "/content/models/baseline_linear_regression.pkl")
joblib.dump(rf, "/content/models/baseline_random_forest.pkl")
joblib.dump(gb, "/content/models/baseline_gradient_boosting.pkl")

print("Saved baseline comparison and models.")


## Baseline Summary

This notebook established a reproducible baseline for RUL prediction on FD001 using engine-level train/validation splits, the official test set with true RUL labels, and a standard RUL-capping strategy. Three baseline regressors — Linear Regression, Random Forest, and Gradient Boosting — were trained on the same feature set and compared using RMSE, MAE, and R² on both a held-out validation split and the official test set.

These results serve as the reference point for `03_advanced_ml_modeling.ipynb`, which will explore hyperparameter tuning, XGBoost, and additional feature engineering (e.g. rolling statistics, degradation-stage indicators) to improve on this baseline. Sequential modeling with LSTM/GRU architectures, which can learn temporal degradation patterns directly from cycle sequences rather than single-cycle snapshots, is covered separately in `04_deep_learning_rul_prediction.ipynb`.